<a href="https://colab.research.google.com/github/NxrFesdac/bourbaki-nlp-avanzado/blob/main/modulo3/Evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from openai import OpenAI
import getpass

# ==========================================
# 1. CONFIGURACIÓN Y CONTEXTO
# ==========================================

api_key = getpass.getpass("Introduce tu OpenAI API Key: ")

# Configura tu cliente de OpenAI
client = OpenAI(api_key=api_key)

pregunta = "¿Cómo solicito mis días de teletrabajo en GlobalCorp?"

respuesta_rag = "Puedes pedir tus 2 días de teletrabajo directamente en el portal WorkLife."

# El "Ground Truth" (La respuesta ideal y perfecta hecha por un humano)
ground_truth = "GlobalCorp permite 2 días de teletrabajo a la semana. Las solicitudes se deben hacer a través del portal WorkLife con al menos 48 horas de antelación."

print("--- DATOS DE LA EVALUACIÓN ---")
print(f"Pregunta: {pregunta}")
print(f"Ground Truth: {ground_truth}")
print(f"Respuesta del RAG: {respuesta_rag}\n")

Introduce tu OpenAI API Key: ··········
--- DATOS DE LA EVALUACIÓN ---
Pregunta: ¿Cómo solicito mis días de teletrabajo en GlobalCorp?
Ground Truth: GlobalCorp permite 2 días de teletrabajo a la semana. Las solicitudes se deben hacer a través del portal WorkLife con al menos 48 horas de antelación.
Respuesta del RAG: Puedes pedir tus 2 días de teletrabajo directamente en el portal WorkLife.



In [ ]:
# ==========================================
# 2. EVALUACIÓN MÉTODO 1: SIMILITUD DEL COSENO
# ==========================================

def obtener_embedding(texto):
    respuesta = client.embeddings.create(
        input=texto,
        model="text-embedding-3-small"
    )
    return respuesta.data[0].embedding

# Convertimos las respuestas a vectores
vector_ground_truth = obtener_embedding(ground_truth)
vector_rag = obtener_embedding(respuesta_rag)

# Calculamos la similitud (producto punto)
similitud = np.dot(vector_ground_truth, vector_rag)

print("--- MÉTODO 1: SIMILITUD MATEMÁTICA ---")
print(f"Similitud del Coseno: {similitud:.4f}")
print("(Nota para la clase: El puntaje será alto porque hablan de lo mismo, ¡pero al RAG le falta información clave! Por eso necesitamos jueces LLM.)\n")


--- MÉTODO 1: SIMILITUD MATEMÁTICA ---
Similitud del Coseno: 0.6985
(Nota para la clase: El puntaje será alto porque hablan de lo mismo, ¡pero al RAG le falta información clave! Por eso necesitamos jueces LLM.)



In [ ]:

# ==========================================
# 3. EVALUACIÓN MÉTODO 2: LLM COMO JUEZ (CATEGÓRICO)
# ==========================================

def evaluar_con_llm_categorico(pregunta, ground_truth, respuesta_evaluar, modelo="gpt-3.5-turbo"):
    prompt_evaluador = f"""
    Eres un evaluador experto. Tu trabajo es comparar una 'Respuesta Generada'
    con una 'Respuesta Ideal' y asignarle una sola categoría.

    Categorías permitidas:
    - CORRECTO: La respuesta generada contiene toda la información clave de la respuesta ideal.
    - PARCIAL: La respuesta generada es cierta, pero omite detalles importantes de la respuesta ideal.
    - INCORRECTO: La respuesta generada contradice la respuesta ideal o es irrelevante.

    Pregunta del usuario: {pregunta}
    Respuesta Ideal (Ground Truth): {ground_truth}
    Respuesta Generada a evaluar: {respuesta_evaluar}

    Devuelve tu evaluación usando EXACTAMENTE este formato:
    RESULTADO: [CORRECTO / PARCIAL / INCORRECTO]
    RAZON: [tu razón breve]
    """

    respuesta = client.chat.completions.create(
        model=modelo,
        messages=[{"role": "user", "content": prompt_evaluador}],
        temperature=0.0 # Temperatura 0 para que no sea creativo al evaluar
    )

    return respuesta.choices[0].message.content

print("--- MÉTODO 2: JUEZ LLM ÚNICO ---")
resultado_juez = evaluar_con_llm_categorico(pregunta, ground_truth, respuesta_rag)
print(resultado_juez)
print("\n")


--- MÉTODO 2: JUEZ LLM ÚNICO ---
RESULTADO: PARCIAL
RAZON: La respuesta generada menciona la cantidad de días de teletrabajo permitidos y la plataforma para solicitarlos, pero omite la información importante de que se deben solicitar con al menos 48 horas de antelación.




In [ ]:
# ==========================================
# 4. EVALUACIÓN MÉTODO 3: PANEL DE JUECES (VOTACIÓN)
# ==========================================
def evaluacion_multi_modelo(pregunta, ground_truth, respuesta_evaluar):
    # Definimos los diferentes "jueces" (modelos de OpenAI)
    modelos_jueces = ["gpt-3.5-turbo", "gpt-4.1-mini", "gpt-4o"]
    votos = []

    # Un prompt único y estricto para todos
    prompt_juez = f"""
    Eres un juez evaluando una 'Respuesta Generada' frente a una 'Respuesta Ideal'.
    Tu trabajo es decidir si la respuesta es APROBADA o RECHAZADA.

    Criterio: Vota APROBADO si la respuesta generada tiene la información central correcta.
    Vota RECHAZADO si falta información crítica o contiene información falsa.

    Pregunta: {pregunta}
    Respuesta Ideal: {ground_truth}
    Respuesta Generada: {respuesta_evaluar}

    Devuelve ÚNICAMENTE una palabra: APROBADO o RECHAZADO.
    """

    print("--- INICIANDO VOTACIÓN MULTI-MODELO ---")

    # Hacemos que cada modelo vote
    for modelo in modelos_jueces:
        respuesta = client.chat.completions.create(
            model=modelo,
            messages=[{"role": "user", "content": prompt_juez}],
            temperature=0.0 # Queremos la respuesta más determinista de cada modelo
        )

        # Limpiamos la respuesta por si el modelo añade un punto final
        voto_crudo = respuesta.choices[0].message.content.strip().upper()
        voto_final = "APROBADO" if "APROBADO" in voto_crudo else "RECHAZADO"

        print(f"Voto de {modelo}: {voto_final}")
        votos.append(voto_final)


    # Calculamos el consenso matemático
    votos_aprobados = votos.count("APROBADO")
    votos_rechazados = len(votos) - votos_aprobados

    print("\n--- RESULTADO DEL CONSENSO ---")
    if votos_aprobados > votos_rechazados:
        veredicto = "APROBADO"
    else:
        veredicto = "RECHAZADO"

    print(f"VEREDICTO FINAL: {veredicto} ({votos_aprobados} a favor, {votos_rechazados} en contra)")
    return veredicto

resultado_final = evaluacion_multi_modelo(pregunta, ground_truth, respuesta_rag)

--- INICIANDO VOTACIÓN MULTI-MODELO ---
Voto de gpt-3.5-turbo: RECHAZADO
Voto de gpt-4.1-mini: RECHAZADO
Voto de gpt-4o: RECHAZADO

--- RESULTADO DEL CONSENSO ---
VEREDICTO FINAL: RECHAZADO (0 a favor, 3 en contra)


In [ ]:
# ==========================================
# 5. EVALUACIÓN MÉTODO 4: JUECES AISLADOS (LLAMADAS SEPARADAS)
# ==========================================

def evaluacion_por_consenso_aislado(pregunta, ground_truth, respuesta_evaluar, modelo="gpt-4o-mini"):

    # Creamos una plantilla base para no repetir las variables en cada juez
    datos_evaluacion = f"""
    Pregunta: {pregunta}
    Respuesta Ideal: {ground_truth}
    Respuesta Generada: {respuesta_evaluar}

    Devuelve ÚNICAMENTE una palabra de estas dos opciones: APROBADO o RECHAZADO.
    """

    # Definimos las instrucciones individuales de cada juez
    jueces = {
        "Juez 1 (Estricto)": f"Eres un juez implacable. Vota RECHAZADO si a la Respuesta Generada le falta CUALQUIER detalle presente en la Respuesta Ideal.\n{datos_evaluacion}",
        "Juez 2 (Empático)": f"Eres un juez empático. Vota APROBADO si la Respuesta Generada es útil, responde la intención principal y no dice mentiras, incluso si omite detalles menores.\n{datos_evaluacion}",
        "Juez 3 (Lógico)": f"Eres un juez puramente lógico. Vota APROBADO solo si el concepto o proceso principal de la Respuesta Generada no contradice la Respuesta Ideal.\n{datos_evaluacion}"
    }

    votos = []

    print("--- INICIANDO VOTACIÓN POR JUECES AISLADOS ---")

    # Hacemos una llamada a la API por cada juez
    for nombre_juez, prompt_juez in jueces.items():
        respuesta = client.chat.completions.create(
            model=modelo,
            messages=[{"role": "user", "content": prompt_juez}],
            temperature=0.0 # Temperatura 0 para evitar alucinaciones en el formato
        )

        # Limpiamos el texto por si el LLM añade puntuación extra
        voto_crudo = respuesta.choices[0].message.content.strip().upper()
        voto_limpio = "APROBADO" if "APROBADO" in voto_crudo else "RECHAZADO"

        votos.append(voto_limpio)
        print(f"{nombre_juez} vota: {voto_limpio}")

    # Calculamos el ganador (El que tenga la mayoría de votos)
    votos_aprobados = votos.count("APROBADO")
    votos_rechazados = votos.count("RECHAZADO")

    veredicto_final = "APROBADO" if votos_aprobados > votos_rechazados else "RECHAZADO"

    print("-" * 30)
    print(f"VEREDICTO FINAL: {veredicto_final} ({votos_aprobados} a favor, {votos_rechazados} en contra)")

    return veredicto_final

resultado_aislado = evaluacion_por_consenso_aislado(pregunta, ground_truth, respuesta_rag)

--- INICIANDO VOTACIÓN POR JUECES AISLADOS ---
Juez 1 (Estricto) vota: RECHAZADO
Juez 2 (Empático) vota: APROBADO
Juez 3 (Lógico) vota: APROBADO
------------------------------
VEREDICTO FINAL: APROBADO (2 a favor, 1 en contra)
